# Calcutta Historical Data Exploration

Notebook scaffold for analyzing normalized historical auction CSVs.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
base = Path.cwd().resolve()
normalized_csv = base / '..' / '..' / 'data' / 'calcutta-historical' / 'calcutta_auction_normalized.csv'
order_csv = base / '..' / '..' / 'data' / 'calcutta-historical' / 'calcutta_auction_order.csv'


In [3]:
normalized = pd.read_csv(normalized_csv)
auction_order = pd.read_csv(order_csv)

In [4]:
normalized.tail()

,year,region,seed,team_name,winning_bid,owner_name,notes,team_still_alive,cashed_1st_round,payout_round1_bonus,...,champs,biggest_loser,total,profit_loss,roi,playin_flag,playin_team_a,playin_team_b,source_section,source_row
123,2024,East,1,UConn,31000,"Kent, Kenny + Company",NaN,Y,NaN,0.0,...,0.0,0.0,0.0,NaN,-1.0,N,NaN,NaN,results_table_2024,86
124,2024,East,5,San Diego State,6200,McManama,NaN,Y,NaN,0.0,...,0.0,0.0,0.0,NaN,-1.0,N,NaN,NaN,results_table_2024,87
125,2024,East,14,Morehead State,3400,Tom MacDonald,Morehead State + 12.5,Y,NaN,0.0,...,0.0,0.0,0.0,NaN,-1.0,N,NaN,NaN,results_table_2024,88
126,2024,East,9,Northwestern,3400,Tom MacDonald,NaN,Y,NaN,0.0,...,0.0,0.0,0.0,NaN,-1.0,N,NaN,NaN,results_table_2024,89
127,2024,East,6,BYU,6200,Tom MacDonald,NaN,Y,NaN,0.0,...,0.0,0.0,0.0,NaN,-1.0,N,NaN,NaN,results_table_2024,90


### Master Table (Canonical Team IDs)
Build master tables by joining Calcutta rows to tournament stages using `year + team_id` from `calcutta_with_team_ids.csv`.


In [9]:
mapped_csv = base / '..' / '..' / 'data' / 'calcutta-historical' / 'calcutta_with_team_ids.csv'
stages_csv = base / '..' / '..' / 'data' / 'tourney-results' / 'm_team_year_stages.csv'
seeds_csv = base / '..' / '..' / 'data' / 'march-machine-learning-mania-2026' / 'MNCAATourneySeeds.csv'
teams_csv = base / '..' / '..' / 'data' / 'march-machine-learning-mania-2026' / 'MTeams.csv'

mapped = pd.read_csv(mapped_csv)
stages = pd.read_csv(stages_csv)
seeds = pd.read_csv(seeds_csv)
teams = pd.read_csv(teams_csv)

mapped = mapped[mapped['year'].astype(str).isin(['2024', '2025'])].copy()
mapped['year'] = pd.to_numeric(mapped['year'], errors='coerce').astype('Int64')
mapped['seed'] = pd.to_numeric(mapped['seed'], errors='coerce')
mapped['winning_bid'] = pd.to_numeric(mapped['winning_bid'], errors='coerce')
for col in ['team_id_primary', 'team_id_a', 'team_id_b']:
    mapped[col] = pd.to_numeric(mapped[col], errors='coerce').astype('Int64')

stages['season'] = pd.to_numeric(stages['season'], errors='coerce').astype('Int64')
stages['team_id'] = pd.to_numeric(stages['team_id'], errors='coerce').astype('Int64')

stage_cols = list(stages.columns)
stage_cols_renamed = {c: f'stage_{c}' for c in stage_cols}
stages_renamed = stages.rename(columns=stage_cols_renamed)

stage_flag_cols = ['won_playin', 'reached_r64', 'reached_r32', 'reached_s16', 'reached_e8', 'reached_f4', 'reached_f2', 'won_championship']
stages_flags_only = stages[['season', 'team_id'] + stage_flag_cols].copy()

# Resolve combo rows to the actual advancing team (usually play-in winner).
combo_a = mapped[['year', 'team_name', 'team_id_a']].dropna(subset=['team_id_a']).copy()
combo_b = mapped[['year', 'team_name', 'team_id_b']].dropna(subset=['team_id_b']).copy()

combo_a = combo_a.merge(
    stages_flags_only,
    how='left',
    left_on=['year', 'team_id_a'],
    right_on=['season', 'team_id'],
)
combo_b = combo_b.merge(
    stages_flags_only,
    how='left',
    left_on=['year', 'team_id_b'],
    right_on=['season', 'team_id'],
)

combo_pick = mapped[['year', 'team_name', 'team_id_primary', 'team_id_a', 'team_id_b']].copy()
combo_pick = combo_pick.merge(
    combo_a[['year', 'team_name'] + stage_flag_cols],
    how='left',
    on=['year', 'team_name'],
)
combo_pick = combo_pick.merge(
    combo_b[['year', 'team_name'] + stage_flag_cols],
    how='left',
    on=['year', 'team_name'],
    suffixes=('_a', '_b'),
)

for col in stage_flag_cols:
    combo_pick[f'{col}_a'] = pd.to_numeric(combo_pick[f'{col}_a'], errors='coerce').fillna(0)
    combo_pick[f'{col}_b'] = pd.to_numeric(combo_pick[f'{col}_b'], errors='coerce').fillna(0)

score_a = (
    combo_pick['won_playin_a'] * 100
    + combo_pick['reached_r64_a'] * 50
    + combo_pick['reached_r32_a'] * 25
    + combo_pick['reached_s16_a'] * 12
    + combo_pick['reached_e8_a'] * 6
    + combo_pick['reached_f4_a'] * 3
    + combo_pick['reached_f2_a'] * 2
    + combo_pick['won_championship_a']
)
score_b = (
    combo_pick['won_playin_b'] * 100
    + combo_pick['reached_r64_b'] * 50
    + combo_pick['reached_r32_b'] * 25
    + combo_pick['reached_s16_b'] * 12
    + combo_pick['reached_e8_b'] * 6
    + combo_pick['reached_f4_b'] * 3
    + combo_pick['reached_f2_b'] * 2
    + combo_pick['won_championship_b']
)

combo_pick['team_id_effective'] = combo_pick['team_id_primary']
needs_combo = combo_pick['team_id_effective'].isna()
combo_pick.loc[needs_combo & (score_a >= score_b), 'team_id_effective'] = combo_pick.loc[needs_combo & (score_a >= score_b), 'team_id_a']
combo_pick.loc[needs_combo & (score_b > score_a), 'team_id_effective'] = combo_pick.loc[needs_combo & (score_b > score_a), 'team_id_b']

primary = mapped.merge(
    combo_pick[['year', 'team_name', 'team_id_effective']],
    how='left',
    on=['year', 'team_name'],
)

primary = primary.merge(
    stages_renamed,
    how='left',
    left_on=['year', 'team_id_effective'],
    right_on=['stage_season', 'stage_team_id'],
)

# Pull official tournament team name + seed from Kaggle IDs for display.
teams_lookup = teams[['TeamID', 'TeamName']].rename(columns={'TeamID': 'team_id_effective', 'TeamName': 'team_name_tourney'})
seeds_lookup = seeds[['Season', 'TeamID', 'Seed']].rename(columns={'Season': 'year', 'TeamID': 'team_id_effective', 'Seed': 'seed_tourney_code'})
seeds_lookup['year'] = pd.to_numeric(seeds_lookup['year'], errors='coerce').astype('Int64')
seeds_lookup['team_id_effective'] = pd.to_numeric(seeds_lookup['team_id_effective'], errors='coerce').astype('Int64')
seeds_lookup['seed_tourney'] = pd.to_numeric(seeds_lookup['seed_tourney_code'].astype(str).str.extract(r'(\d{2})')[0], errors='coerce')

primary = primary.merge(teams_lookup, how='left', on='team_id_effective')
primary = primary.merge(seeds_lookup[['year', 'team_id_effective', 'seed_tourney_code', 'seed_tourney']], how='left', on=['year', 'team_id_effective'])

primary['team_name_effective'] = primary['team_name_tourney'].fillna(primary['team_name'])
primary['seed_effective'] = primary['seed_tourney'].fillna(primary['seed'])

calcutta_base_cols = [
    'team_name_effective', 'seed_effective', 'region', 'year', 'winning_bid', 'owner_name',
    'team_id_effective', 'team_id_primary', 'team_id_a', 'team_id_b',
]

master_table_primary = primary[calcutta_base_cols + [f'stage_{c}' for c in stage_cols]].copy()
master_table_primary = master_table_primary.rename(columns={'team_name_effective': 'team_name', 'seed_effective': 'seed'})

print(f'Primary table rows: {len(master_table_primary)}')
print(f'Primary stage join rate: {master_table_primary["stage_team_id"].notna().mean():.1%}')

master_table_primary[master_table_primary['team_id_a'].notna()]


Primary table rows: 128
Primary stage join rate: 100.0%


,team_name,seed,region,year,winning_bid,owner_name,team_id_effective,team_id_primary,team_id_a,team_id_b,...,stage_team_name,stage_reached_playin,stage_won_playin,stage_reached_r64,stage_reached_r32,stage_reached_s16,stage_reached_e8,stage_reached_f4,stage_reached_f2,stage_won_championship
4,Xavier,11,Midwest,2025,4000,UNKNOWN,1462,<NA>,1400,1462,...,Xavier,1,1,1,0,0,0,0,0,0
20,North Carolina,11,South,2025,4600,UNKNOWN,1314,<NA>,1361,1314,...,North Carolina,1,1,1,0,0,0,0,0,0
22,Alabama St,16,South,2025,4000,UNKNOWN,1106,<NA>,1106,1384,...,Alabama St,1,1,1,0,0,0,0,0,0
53,Mt St Mary's,16,East,2025,6000,UNKNOWN,1291,1291,1110,1291,...,Mt St Mary's,1,1,1,0,0,0,0,0,0
70,Wagner,16,West,2024,2100,Kulhman + Girardin,1447,<NA>,1224,1447,...,Wagner,1,1,1,0,0,0,0,0,0
81,Colorado St,10,Midwest,2024,2900,Moncrief + Sparty,1161,<NA>,1438,1161,...,Colorado St,1,1,1,0,0,0,0,0,0
86,Grambling,16,Midwest,2024,3000,Moncrief + Sparty,1212,<NA>,1286,1212,...,Grambling,1,1,1,0,0,0,0,0,0
97,Colorado,10,South,2024,5200,"Kent, Kenny + Company",1160,<NA>,1129,1160,...,Colorado,1,1,1,1,0,0,0,0,0


Evan Miya

### 3) Read Files
Load tournament stage outcomes and payout rule config.


In [6]:
evan_miya_csv = base / '..' / '..' / 'data' / 'kaggle' / 'EvanMiya.csv'
evan_miya = pd.read_csv(evan_miya_csv)
evan_miya.head()


FileNotFoundError: [Errno 2] No such file or directory: '/Users/harrywang/Desktop/Projects/march-madness-calcutta/src/data-exploration/../../data/kaggle/EvanMiya.csv'

In [ ]:
normalized_for_join = normalized.copy()
normalized_for_join['join_year'] = normalized_for_join['year'].astype(str).str.strip()
normalized_for_join['join_team'] = normalized_for_join['team_name'].astype(str).str.strip().str.lower()

evan_for_join = evan_miya.copy()
evan_for_join['join_year'] = evan_for_join['YEAR'].astype(str).str.strip()
evan_for_join['join_team'] = evan_for_join['TEAM'].astype(str).str.strip().str.lower()

calcutta_with_evan = normalized_for_join.merge(
    evan_for_join,
    how='left',
    on=['join_year', 'join_team'],
    suffixes=('', '_evan')
)

match_rate = calcutta_with_evan['TEAM'].notna().mean()
print(f'Joined rows: {len(calcutta_with_evan)}')
print(f'Match rate: {match_rate:.1%}')

calcutta_with_evan[['year', 'team_name', 'region', 'seed', 'winning_bid', 'TEAM', 'RELATIVE RATING']].head(20)
